In [15]:
import pandas as pd

df_clean = pd.read_parquet('../data/curated/part1_taxi_curated.parquet')
df_clean.sample(5, random_state=42)


,PULocationID,DOLocationID,PU_borough,PU_zone,DO_borough,DO_zone,tpep_pickup_datetime,tpep_dropoff_datetime,pickup_date,pickup_hour,...,trip_distance,fare_amount,tip_amount,total_amount,payment_type,trip_duration_min,speed_mph,fare_per_mile,tip_rate,distance_bucket
2409739,237,48,Manhattan,Upper East Side South,Manhattan,Clinton East,2023-01-26 22:42:40,2023-01-26 22:55:51,2023-01-26,22,...,2.10,14.2,3.84,23.04,1,13.183333,9.557522,6.761905,0.166667,"[1,3)"
1442295,170,161,Manhattan,Murray Hill,Manhattan,Midtown Center,2023-01-17 08:33:17,2023-01-17 08:42:16,2023-01-17,8,...,0.83,9.3,2.66,15.96,1,8.983333,5.543599,11.204819,0.166667,"[0,1)"
2803684,43,161,Manhattan,Central Park,Manhattan,Midtown Center,2023-01-31 09:26:42,2023-01-31 09:36:13,2023-01-31,9,...,1.87,11.4,0.00,15.40,2,9.516667,11.789842,6.096257,0.000000,"[1,3)"
1988338,148,114,Manhattan,Lower East Side,Manhattan,Greenwich Village South,2023-01-22 14:35:04,2023-01-22 14:48:41,2023-01-22,14,...,1.26,12.8,3.36,20.16,1,13.616667,5.552020,10.158730,0.166667,"[1,3)"
1052786,158,48,Manhattan,Meatpacking/West Village West,Manhattan,Clinton East,2023-01-13 00:20:57,2023-01-13 00:30:25,2023-01-13,0,...,2.69,13.5,1.00,19.50,1,9.466667,17.049296,5.018587,0.051282,"[1,3)"


## A. Required groupby metrics.

In [ ]:
# --- A. Non-trivial statistics  ---
# 1. Daily trip counts grouped by PU_borough and pickup_date.
daily_trip_counts = (
    df_clean.groupby(['PU_borough', 'pickup_date'])
    .size()
    .reset_index(name='trip_count')
)

print("--- Daily Trip Counts ---")
daily_trip_counts


--- Daily Trip Counts ---


,PU_borough,pickup_date,trip_count
0,Bronx,2023-01-01,91
1,Bronx,2023-01-02,61
2,Bronx,2023-01-03,122
3,Bronx,2023-01-04,165
4,Bronx,2023-01-05,140
...,...,...,...
212,Unknown,2023-01-27,1329
213,Unknown,2023-01-28,1204
214,Unknown,2023-01-29,1282
215,Unknown,2023-01-30,1089


In [22]:
# 2. Mean and median of fare_amount and trip_distance grouped by PU_borough and weekday

fare_dist_stats = (
    df_clean.groupby(["PU_borough", "weekday"])[["fare_amount", "trip_distance"]]
    .agg(["mean", "median"])
    .reset_index()
)

print("--- Fare and Distance Stats ---")
fare_dist_stats.head()

--- Fare and Distance Stats ---


PU_borough weekday fare_amount        trip_distance       
                            mean median          mean median
0      Bronx     Fri   31.846877   26.2      7.271738   4.69
1      Bronx     Mon   31.415012   29.2      7.375206   5.70
2      Bronx     Sat   27.605784   22.5      6.652276   4.25
3      Bronx     Sun   26.862102   22.5      5.856306   4.40
4      Bronx     Thu   29.725138   25.4      6.898687   4.70

In [24]:
# 3. For each pickup_hour, the proportion of trips by payment_type
payment_type_props = (
    df_clean.groupby("pickup_hour")["payment_type"]
    .value_counts(normalize=True)
    .reset_index(name="proportion")
)

print("\n--- Payment Type Proportions ---")
payment_type_props


--- Payment Type Proportions ---


,pickup_hour,payment_type,proportion
0,0,1,0.829961
1,0,2,0.157484
2,0,4,0.008282
3,0,3,0.004273
4,1,1,0.841938
...,...,...,...
91,22,3,0.003122
92,23,1,0.838370
93,23,2,0.151015
94,23,4,0.007126


In [25]:
# 4. Mean tip_rate grouped by distance_bucket
mean_tip_rate_by_distance = (
    df_clean.groupby("distance_bucket")["tip_rate"]
    .mean()
    .reset_index(name="mean_tip_rate")
)

print("\n--- Mean Tip Rate by Distance Bucket ---")
mean_tip_rate_by_distance


--- Mean Tip Rate by Distance Bucket ---


,distance_bucket,mean_tip_rate
0,"[0,1)",0.119133
1,"[1,3)",0.123694
2,"[3,5)",0.120474
3,"[5,10)",0.112818
4,"[10,+)",0.111043


## B. Required window-style metrics.

In [26]:
daily_trip_counts = daily_trip_counts.sort_values(
    by=["PU_borough", "pickup_date"]
).reset_index(drop=True)

daily_trip_counts

,PU_borough,pickup_date,trip_count
0,Bronx,2023-01-01,91
1,Bronx,2023-01-02,61
2,Bronx,2023-01-03,122
3,Bronx,2023-01-04,165
4,Bronx,2023-01-05,140
...,...,...,...
212,Unknown,2023-01-27,1329
213,Unknown,2023-01-28,1204
214,Unknown,2023-01-29,1282
215,Unknown,2023-01-30,1089


In [31]:
# 1. For each PU_borough, a 7-day rolling mean of daily trip counts ordered by pickup_date
daily_trip_counts["rolling_7d_mean"] = (daily_trip_counts
    .groupby("PU_borough")[
        "trip_count"
    ].transform(lambda x: x.rolling(window=7, min_periods=1)
    .mean()
))

daily_trip_counts

,PU_borough,pickup_date,trip_count,rolling_7d_mean
0,Bronx,2023-01-01,91,91.000000
1,Bronx,2023-01-02,61,76.000000
2,Bronx,2023-01-03,122,91.333333
3,Bronx,2023-01-04,165,109.750000
4,Bronx,2023-01-05,140,115.800000
...,...,...,...,...
212,Unknown,2023-01-27,1329,1165.428571
213,Unknown,2023-01-28,1204,1171.142857
214,Unknown,2023-01-29,1282,1175.428571
215,Unknown,2023-01-30,1089,1180.571429


In [ ]:
# 2. For each PU_borough, the day-over-day difference and day-over-day percent change
daily_trip_counts["dod_diff"] = daily_trip_counts.groupby("PU_borough")[
    "trip_count"
].diff()


daily_trip_counts["dod_pct_change"] = daily_trip_counts.groupby("PU_borough")[
    "trip_count"
].pct_change()

In [32]:
print(daily_trip_counts[daily_trip_counts['PU_borough'] == 'Manhattan'].head(10))

   PU_borough pickup_date  trip_count  rolling_7d_mean
87  Manhattan  2022-12-31          22        22.000000
88  Manhattan  2023-01-01       59326     29674.000000
89  Manhattan  2023-01-02       50246     36531.333333
90  Manhattan  2023-01-03       67722     44329.000000
91  Manhattan  2023-01-04       77910     51045.200000
92  Manhattan  2023-01-05       84106     56555.333333
93  Manhattan  2023-01-06       86722     60864.857143
94  Manhattan  2023-01-07       89931     73709.000000
95  Manhattan  2023-01-08       68575     75030.285714
96  Manhattan  2023-01-09       69456     77774.571429


## C. Required EDA questions (answer all three with evidence).

1. Which borough has the strongest weekday effect? Quantify it using:

$$
S_{borough} = \frac{\max({\mu_{\text{weekday,borough}}})-{\min({\mu_{\text{weekday,borough}}})}}{\mu_{\text{borough}}}
$$

where $S_{\text{borough}}$ is the measure of weekday effect , µweekday, borough is mean daily trips for that weekday in the borough and µweekday, borough is mean daily trips for the borough.